# Projet NLP : entraîner un tokeniseur BPE et le comparer à d’autres découpages

**Master 2 MIASHS, Université Lumière Lyon 2 (2025)** · cours « Le NLP pour les sciences sociales »
**Auteurs :** Wendyam Ariel Clauvis TIEMTORE et Said Mmadi Mohamed

**Objectif.** Créer un tokeniseur *Byte-Pair Encoding* (BPE) à partir de zéro avec la bibliothèque `tokenizers`, l’entraîner sur un petit corpus, puis comparer son découpage à trois approches classiques : `split(' ')`, NLTK et spaCy.

**Étapes.** Corpus d’exemple → tokeniseur BPE vide → normalisation (NFD, suppression des accents, minuscules) → pré-tokenisation par espaces → entraînement → encodage, décodage et sauvegarde → comparaison.

## 1. Tokeniseur BPE : création, entraînement et test

In [ ]:
# Installation des bibliothèques nécessaires
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from tokenizers.decoders import BPEDecoder
from tokenizers.normalizers import NFD, StripAccents, Lowercase, Sequence

# Préparation d'un corpus d'exemple
corpus = [
    "J'ai un chat",
    "Mon chat est très mignon",
    "Le chat s'est assis sur le tapis",
    "Il y a aussi un chien"
]

# Création d'un tokenizer BPE vide avec la classe `Tokenizer`
tokenizer = Tokenizer(models.BPE())

# Configuration de la normalisation du texte
tokenizer.normalizer = Sequence([NFD(), StripAccents(), Lowercase()])

# Configuration du pré-tokeniseur (séparation des mots)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Entraînement du tokenizer BPE sur le corpus d'exemple
trainer = trainers.BpeTrainer(vocab_size=1000, show_progress=True, special_tokens=["<unk>", "<s>", "</s>", "<pad>"])

# Entraînement du tokenizer avec le corpus
tokenizer.train_from_iterator(corpus, trainer)

# Décodeur pour transformer les indices des tokens en texte
tokenizer.decoder = BPEDecoder()

# Test du tokenizer
text = "Le chat s'est assis sur le tapis"
encoded = tokenizer.encode(text)
print("Encoded:", encoded.tokens)

# Décodeur des tokens pour retrouver le texte original
decoded = tokenizer.decode(encoded.ids)
print("Decoded:", decoded)

# Sauvegarde du tokenizer pour réutilisation
tokenizer.save("bpe_tokenizer.json")

Encoded: ['le', 'chat', 's', "'", 'est', 'assis', 'sur', 'le', 'tapis']
Decoded: lechats'estassissurletapis


## 2. Découpage simple par espaces

In [ ]:
text = "Le renard brun rapide saute par-dessus le chien paresseux."
tokens_simple = text.split(' ')
print("Simple Split:", tokens_simple)

Simple Split: ['Le', 'renard', 'brun', 'rapide', 'saute', 'par-dessus', 'le', 'chien', 'paresseux.']


## 3. Tokenisation NLTK

In [ ]:
import nltk

# Téléchargement des données 'point_loss'
nltk.download('punkt_tab')

# Utilisation du word_tokenize
tokens_nltk = nltk.word_tokenize(text)
print("NLTK Tokenisation:", tokens_nltk)

NLTK Tokenisation: ['Le', 'renard', 'brun', 'rapide', 'saute', 'par-dessus', 'le', 'chien', 'paresseux', '.']


## 4. Tokenisation spaCy

In [ ]:
import spacy

# Chargement du modèle linguistique de base
nlp = spacy.load("en_core_web_sm")

doc = nlp(text)
tokens_spacy = [token.text for token in doc]
print("SpaCy Tokenisation:", tokens_spacy)

SpaCy Tokenisation: ['Le', 'renard', 'brun', 'rapide', 'saute', 'par', '-', 'dessus', 'le', 'chien', 'paresseux', '.']


## 5. Conclusion

Le découpage simple (`split(' ')`) ne gère ni la ponctuation ni les mots composés : « paresseux. » reste un seul token, alors qu’il est utile de traiter « paresseux » et « . » séparément. NLTK et spaCy séparent la ponctuation ; spaCy découpe en plus le trait d’union de « par-dessus ». Pour des tâches qui demandent une séparation nette des mots et de la ponctuation, NLTK et spaCy sont les mieux adaptés à ce corpus.

## Relecture (novembre 2026)

Deux remarques ajoutées à la relecture, le code d’origine ci-dessus restant inchangé :

1. **Le décodage recolle les mots** (« lechats'estassis… ») parce que `BPEDecoder` reconstruit les espaces à partir d’un suffixe de fin de mot, que ni le modèle ni l’entraîneur ne déclaraient. La cellule suivante déclare ce suffixe et retrouve une phrase lisible.
2. **spaCy est chargé avec un modèle anglais** (`en_core_web_sm`) sur une phrase française ; `fr_core_news_sm` serait le choix naturel.

In [ ]:
# Correction : déclarer le suffixe de fin de mot au modèle, à l'entraîneur et au décodeur
tok = Tokenizer(models.BPE(end_of_word_suffix="</w>"))
tok.normalizer = Sequence([NFD(), StripAccents(), Lowercase()])
tok.pre_tokenizer = pre_tokenizers.Whitespace()
tok.train_from_iterator(corpus, trainers.BpeTrainer(vocab_size=1000, show_progress=False, end_of_word_suffix="</w>",
                                                   special_tokens=["<unk>", "<s>", "</s>", "<pad>"]))
tok.decoder = BPEDecoder(suffix="</w>")
enc = tok.encode("Le chat s'est assis sur le tapis")
print("Encoded:", enc.tokens)
print("Decoded:", tok.decode(enc.ids))

Encoded: ['le</w>', 'chat</w>', 's</w>', "'</w>", 'est</w>', 'assis</w>', 'sur</w>', 'le</w>', 'tapis</w>']
Decoded: le chat s ' est assis sur le tapis
